<a href="https://colab.research.google.com/github/Hasnaincoder1/Flyrankrepo1/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hasnaincoder1/Flyrankrepo1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Using the actual paper I was given (FlyRank, "The State of AI-Driven SEO," March 2026 edition — 341,701 pages, 57 brands).

**Finding: "What Predicts Health?"** (ML Appendix, Random Forest feature importance) — reports Average Position at 43% importance, Impressions at 32%, Scroll Depth at 15%, CTR at 8%, with everything else (content age, word count, days visible, sessions, AI sessions) near 0%.

**My methodology question:** the paper is admirably upfront that "the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal" — but I'd push that caveat one step further. `Health Score = Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts)`, per the paper's own metric definition. The top four features by importance — Position, Impressions, Scroll Depth, CTR — are not just *correlated* with the label, they are literally its four additive components. This isn't partial construction, it's closer to the model rediscovering the health-score formula from its own ingredients. My question: does this feature-importance ranking tell us anything a reader couldn't get by re-reading the health-score definition itself? I'd want to see feature importance for a Random Forest trained to predict something the paper *doesn't* construct from these same four numbers — say, next-month impression growth — before treating this appendix page as informative rather than circular by design.

**Finding: "The Freshness Multiplier"** (Finding #4) — reports that 365+ day content refreshed within the last 30 days shows a 3.2x health boost (10.7 → 34.5) and 57x more impressions (71 → 4,039) versus otherwise-similar stale old pages.

**My methodology question:** "refreshed" is an editorial choice, not a random assignment, so this is a refreshed-vs-not-refreshed group comparison rather than a controlled experiment. If editors tend to refresh old pages that still show some residual signal worth the effort (a real topic, some surviving backlinks, a keyword still worth targeting) rather than picking uniformly at random from the 365+ pool, part of the 57x impression gap could reflect *which pages got selected for refresh* rather than what the refresh itself caused. To the paper's credit, its own Limitations section says plainly "observational study: correlations do not prove causation" — but the finding's framing ("The Freshness Multiplier," "refresh timing is one of the strongest measured levers available") reads more causally in the body text than that limitations-page caveat fully covers. I'd want to know whether the comparison group was matched on pre-refresh trajectory (were these pages already trending up before the refresh, or genuinely flat/declining?) before treating 57x as the effect of refreshing itself.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5 model was already trained on a grouped-by-client split, so the "before" here is what I'd have gotten if I'd taken the easier, less honest path: a plain random row-level split, which lets the same client show up in both train and test. "After" is the grouped split I actually used. Same features, same model (Logistic Regression), same metric (precision@50, AUC) — only the split changes.

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
for c in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{c}"] = np.log1p(df[c])

NUMERIC = ["search_volume", "competition", "cpc", "word_count", "char_count",
           "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
           "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
           "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"]
CATEGORICAL = ["competition_level", "content_type", "main_intent", "age_tier",
               "freshness_tier", "word_count_tier", "impression_tier"]

X = df[NUMERIC + CATEGORICAL]
y = df["is_declining_label"]
groups = df["client_id"]

pre = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL),
])

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# BEFORE -- random row-level split (client can leak across train/test)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
pipe_random = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, random_state=42))]).fit(X_tr, y_tr)
proba_random = pipe_random.predict_proba(X_te)[:, 1]
p50_random = precision_at_k(proba_random, y_te.values, 50)
auc_random = roc_auc_score(y_te, proba_random)

# AFTER -- grouped split by client_id (same as w05_model.ipynb)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups))
pipe_grouped = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, random_state=42))]).fit(X.iloc[tr_idx], y.iloc[tr_idx])
proba_grouped = pipe_grouped.predict_proba(X.iloc[te_idx])[:, 1]
p50_grouped = precision_at_k(proba_grouped, y.iloc[te_idx].values, 50)
auc_grouped = roc_auc_score(y.iloc[te_idx], proba_grouped)

comparison = pd.DataFrame([
    {"split": "BEFORE -- random row-level (client leaks across train/test)", "precision@50": round(p50_random, 3), "auc": round(auc_random, 3)},
    {"split": "AFTER -- grouped by client_id (honest, matches w05)",          "precision@50": round(p50_grouped, 3), "auc": round(auc_grouped, 3)},
])
print(f"gap: precision@50 drops {p50_random - p50_grouped:+.3f}, AUC drops {auc_random - auc_grouped:+.3f}")
print("This gap IS the finding -- it's roughly how much of the random-split score was the model")
print("partly recognizing a client it had already seen, rather than generalizing to a new one.")
comparison

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/content_refresh_anonymized.csv'

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Per `hunting-leakage-and-validating`'s test: train once WITHOUT a suspect column, once WITH it, and see if the score collapses back down when removed. I test two suspects that are excluded from my real feature set (see ML-08 §2): `trend_pct` (the label's own definition) and `impressions_last_30d`/`impressions_prev_30d` (the two raw numbers the label's threshold is computed from).


In [ ]:
def audit_run(numeric_cols, label_text):
    Xa = df[numeric_cols + CATEGORICAL]
    pre_a = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_cols),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL),
    ])
    tr_i, te_i = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42).split(Xa, y, groups))
    pipe_a = Pipeline([("pre", pre_a), ("clf", LogisticRegression(max_iter=2000, random_state=42))]).fit(Xa.iloc[tr_i], y.iloc[tr_i])
    proba_a = pipe_a.predict_proba(Xa.iloc[te_i])[:, 1]
    auc_a = roc_auc_score(y.iloc[te_i], proba_a)
    print(f"{label_text:55s} AUC = {auc_a:.3f}")
    return auc_a

honest_auc = audit_run(NUMERIC, "WITHOUT suspect (my real w05 feature set)")
leak1_auc = audit_run(NUMERIC + ["trend_pct"], "WITH trend_pct added")
leak2_auc = audit_run(NUMERIC + ["impressions_last_30d", "impressions_prev_30d"], "WITH last_30d/prev_30d added")

print()
print(f"Confession: adding trend_pct jumps AUC from {honest_auc:.3f} to {leak1_auc:.3f} -- textbook")
print(f"label-derived leakage (trend_pct IS how the label is defined). Adding the raw 30d/prev-30d")
print(f"pair jumps it to {leak2_auc:.3f} -- smaller but still a real leak, since those two numbers ARE")
print(f"the label's threshold inputs. Neither suspect is in my real w05/w06 feature set -- both were")
print(f"excluded from the start (ML-08 section 2), and this audit is the receipt proving why that")
print(f"exclusion mattered, not just a formality.")

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (too bold):** "Logistic Regression beats the baseline at identifying declining pages" (from ML-08 §4) — this reads as a settled, general capability claim.

**Rewritten (safe):** On this starter sample, evaluated on 8 clients the model never trained on, Logistic Regression reached a measured precision@50 of 0.76 versus a fair rule-based baseline's 0.30 and a 0.52 base rate — an observed, directional improvement on this specific slice and split. This is decision-support evidence that the approach is worth extending to the full warehouse, not a claim that it identifies decline in general, and not a claim about causality (nothing here says refreshing a flagged page *causes* recovery, only that the flag correlates with the observed label on this sample).


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.